In [2]:
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd


In [54]:
# Check what sheets are available in the Excel file
xl_file = pd.ExcelFile("../data/SPFmicrodata.xlsx")
print("Available indicators:", xl_file.sheet_names)

# Load only the NGDP sheet
df = pd.read_excel("../data/SPFmicrodata.xlsx", sheet_name="NGDP")
print(f"\nLoaded NGDP sheet: {df.shape[0]} rows × {df.shape[1]} columns")
print("\nColumn names:", df.columns.tolist())

Available indicators: ['NGDP', 'PGDP', 'CPROF', 'UNEMP', 'EMP', 'INDPROD', 'HOUSING', 'TBILL', 'BOND', 'BAABOND', 'TBOND', 'RGDP', 'RCONSUM', 'RNRESIN', 'RRESINV', 'RFEDGOV', 'RSLGOV', 'RCBI', 'REXPORT', 'CPI5YR', 'PCE5YR', 'CPI10', 'PCE10', 'RGDP10', 'PROD10', 'STOCK10', 'BOND10', 'BILL10', 'PRGDP', 'PRPGDP', 'PRUNEMP', 'PRCCPI', 'PRCPCE', 'RECESS', 'CPI', 'CORECPI', 'PCE', 'COREPCE', 'UBAR', 'SPR_TBOND_TBILL', 'SPR_BAA_AAA', 'SPR_BAA_TBOND', 'SPR_AAA_TBOND', 'CPIF5', 'PCEF5', 'RR1_TBILL_PGDP', 'RR2_TBILL_PGDP', 'RR3_TBILL_PGDP', 'RR1_TBILL_CPI', 'RR2_TBILL_CPI', 'RR3_TBILL_CPI', 'RR1_TBILL_CCPI', 'RR2_TBILL_CCPI', 'RR3_TBILL_CCPI', 'RR1_TBILL_PCE', 'RR2_TBILL_PCE', 'RR3_TBILL_PCE', 'RR1_TBILL_CPCE', 'RR2_TBILL_CPCE', 'RR3_TBILL_CPCE']

Loaded NGDP sheet: 9145 rows × 12 columns

Column names: ['YEAR', 'QUARTER', 'ID', 'INDUSTRY', 'NGDP1', 'NGDP2', 'NGDP3', 'NGDP4', 'NGDP5', 'NGDP6', 'NGDPA', 'NGDPB']


/Users/Parimah/anaconda3/lib/python3.11/site-packages/openpyxl/worksheet/header_footer.py:48: UserWarning: Cannot parse header or footer so it will be ignored
  warn("""Cannot parse header or footer so it will be ignored""")


In [55]:
df.head()

,YEAR,QUARTER,ID,INDUSTRY,NGDP1,NGDP2,NGDP3,NGDP4,NGDP5,NGDP6,NGDPA,NGDPB
0,1968,4,1,NaN,871.0,884.0,895.0,907.0,920.0,938.0,NaN,NaN
1,1968,4,2,NaN,871.0,891.0,910.0,929.0,958.0,973.0,NaN,NaN
2,1968,4,3,NaN,871.0,883.0,894.0,906.0,924.0,944.0,NaN,NaN
3,1968,4,4,NaN,871.0,885.0,891.0,902.0,919.0,937.0,NaN,NaN
4,1968,4,5,NaN,871.0,895.0,913.0,935.0,940.0,970.0,NaN,NaN


In [56]:
print(" Number of unique IDs:", df['ID'].nunique())
print(" Number of unique years:", df['YEAR'].nunique())
print(" Number of na values in NGDP6:", df['NGDP6'].isna().sum())

 Number of unique IDs: 462
 Number of unique years: 58
 Number of na values in NGDP6: 974


Forecasts for the quarterly and annual level of nominal GDP. Seasonally
adjusted, annual rate, billions $. Prior to 1992, these are forecasts for
nominal GNP. Annual forecasts are for the annual average of the quarterly
levels.

First survey to include this variable: 1968:Q4


https://www.philadelphiafed.org/-/media/FRBP/Assets/Surveys-And-Data/survey-of-professional-forecasters/spf-documentation.pdf?sc_lang=en&hash=8408A4F1BF351A3C268B40F6BC7B95AA


Actual Nominal GDP Data by Quarters 

https://fred.stlouisfed.org/series/GDP

In [57]:
# Check what sheets are available in the Excel file
NGDP_xl_file = pd.ExcelFile("../data/GDP.xlsx")
print("Available indicators:", NGDP_xl_file.sheet_names)

# Load only the NGDP sheet
NGDP_act = pd.read_excel("../data/GDP.xlsx", sheet_name="Quarterly")
# Display the first few rows of the NGDP data
print(NGDP_act.head())

Available indicators: ['README', 'Quarterly']
  observation_date      GDP
0       1947-01-01  243.164
1       1947-04-01  245.968
2       1947-07-01  249.585
3       1947-10-01  259.745
4       1948-01-01  265.742


In [ ]:
# Data for error statistics (projections and realizations) directly for NGDP. 
err_stat = pd.read_excel("../data/Data_SPF_Error_Statistics_NGDP_3_AIC.xls", engine='xlrd')

# Parse Date column (e.g. "1968:04") into YEAR and QUARTER
err_stat[['YEAR', 'QUARTER']] = err_stat['Date'].str.split(':', expand=True).astype(int)
err_stat.drop(columns=['Date'], inplace=True)

print(f"Shape: {err_stat.shape}")
print(f"Columns: {err_stat.columns.tolist()}")
err_stat.head(10)

In [58]:
# Convert the observation_date colum to year and quarter
NGDP_act['YEAR'] = NGDP_act['observation_date'].dt.year
NGDP_act['QUARTER'] = NGDP_act['observation_date'].dt.month.apply(lambda x: (x-1)//3 + 1)
NGDP_act.rename(columns={'NA000334Q': 'NGDP_actual'}, inplace=True)
# only keep the data from 1968 Q3 onwards to match the forecast data
NGDP_act = NGDP_act[(NGDP_act['YEAR'] > 1968) | ((NGDP_act['YEAR'] == 1968) & (NGDP_act['QUARTER'] >= 4))]

NGDP_act.drop(columns=['observation_date'], inplace=True)

NGDP_act.head()

,GDP,YEAR,QUARTER
87,968.030,1968,4
88,993.337,1969,1
89,1009.020,1969,2
90,1029.956,1969,3
91,1038.147,1969,4


In [59]:

# NGDP1 = t-1 (previous quarter historical)
# NGDP2 = t+0 (nowcast, current quarter)
# NGDP3 = t+1, NGDP4 = t+2, NGDP5 = t+3, NGDP6 = t+4
horizon_offsets = {
    'NGDP1': -1,
    'NGDP2':  0,
    'NGDP3':  1,
    'NGDP4':  2,
    'NGDP5':  3,
    'NGDP6':  4,
}

# Build a fast (year, quarter) -> GDP lookup
gdp_lookup = NGDP_act.set_index(['YEAR', 'QUARTER'])['GDP']

errors_df = df[['YEAR', 'QUARTER', 'ID', 'INDUSTRY'] + list(horizon_offsets.keys())].copy()

for col, offset in horizon_offsets.items():
    # Shift survey (YEAR, QUARTER) by offset to get the target quarter
    total    = (errors_df['YEAR'] - 1) * 4 + (errors_df['QUARTER'] - 1) + offset
    t_year   = (total // 4) + 1
    t_qtr    = (total % 4) + 1
    
    gdp_actual = np.array([gdp_lookup.get((y, q), np.nan)
                            for y, q in zip(t_year, t_qtr)])
    
    errors_df[f'error_{col}'] = errors_df[col].values - gdp_actual
    errors_df.drop(columns=[col], inplace=True)

errors_df.head(10)


,YEAR,QUARTER,ID,INDUSTRY,error_NGDP1,error_NGDP2,error_NGDP3,error_NGDP4,error_NGDP5,error_NGDP6
0,1968,4,1,NaN,NaN,-84.03,-98.337,-102.02,-109.956,-100.147
1,1968,4,2,NaN,NaN,-77.03,-83.337,-80.02,-71.956,-65.147
2,1968,4,3,NaN,NaN,-85.03,-99.337,-103.02,-105.956,-94.147
3,1968,4,4,NaN,NaN,-83.03,-102.337,-107.02,-110.956,-101.147
4,1968,4,5,NaN,NaN,-73.03,-80.337,-74.02,-89.956,-68.147
5,1968,4,6,NaN,NaN,-83.03,-98.337,-101.02,-119.956,-113.147
6,1968,4,7,NaN,NaN,-84.03,-98.337,-98.02,-99.956,-89.147
7,1968,4,8,NaN,NaN,-83.03,-105.337,-109.02,-108.956,-102.147
8,1968,4,9,NaN,NaN,-85.03,-100.337,-109.02,-117.956,-108.147
9,1968,4,10,NaN,NaN,-83.03,-98.337,-95.02,-92.956,-81.147


In [60]:
# let's only keep NGDP3 errors for now
errors_df = errors_df[['YEAR', 'QUARTER', 'ID', 'INDUSTRY', 'error_NGDP3']].copy()
errors_df['error_NGDP3'] = errors_df['error_NGDP3']**2
errors_df.rename(columns={'error_NGDP3': 'Squared Error'}, inplace=True)

# reshape the matrix to have one row per (YEAR, QUARTER) and columns for each ID
errors_df = errors_df.pivot_table(index=['YEAR', 'QUARTER'], columns='ID', values='Squared Error')

errors_df.head(10)

ID                     1            2             3             4    \
YEAR QUARTER                                                          
1968 4         9670.165569  6945.055569   9867.839569  10472.861569   
1969 1                 NaN  7228.400400  10205.040400  11029.200400   
     2        10599.937936          NaN   9991.201936  10192.113936   
     3         8307.775609          NaN   8307.775609           NaN   
     4         7089.640000          NaN   7956.640000   6593.440000   
1970 1         8906.640625          NaN   8718.890625   6951.390625   
     2         9420.449481          NaN   8659.977481   9812.685481   
     3         6179.217664          NaN   6497.649664   8392.025664   
     4        13260.904336          NaN  13492.216336           NaN   
1971 1        12830.319441          NaN           NaN  13752.487441   

ID                    5             6             7             8    \
YEAR QUARTER                                                          
1968 4        6454.033569   9670.165569   9670.165569  11095.883569   
1969 1                NaN   9219.840400   7747.520400  10205.040400   
     2                NaN  11015.761936           NaN   9991.201936   
     3                NaN   8307.775609           NaN           NaN   
     4                NaN   7089.640000   7259.040000   7259.040000   
1970 1                NaN  10686.390625   9481.890625   9481.890625   
     2                NaN  10621.157481  11037.393481  11037.393481   
     3                NaN   6179.217664   8209.809664   7328.729664   
     4                NaN           NaN  13960.840336           NaN   
1971 1                NaN           NaN  13988.029441           NaN   

ID                     9            10   ...  600  601  602  603  604  605  \
YEAR QUARTER                             ...                                 
1968 4        10067.513569  9670.165569  ...  NaN  NaN  NaN  NaN  NaN  NaN   
1969 1         9412.880400  9028.800400  ...  NaN  NaN  NaN  NaN  NaN  NaN   
     2        11226.673936  8827.729936  ...  NaN  NaN  NaN  NaN  NaN  NaN   
     3                 NaN  6748.129609  ...  NaN  NaN  NaN  NaN  NaN  NaN   
     4         7089.640000          NaN  ...  NaN  NaN  NaN  NaN  NaN  NaN   
1970 1         8718.890625          NaN  ...  NaN  NaN  NaN  NaN  NaN  NaN   
     2                 NaN  9227.331481  ...  NaN  NaN  NaN  NaN  NaN  NaN   
     3                 NaN  7158.513664  ...  NaN  NaN  NaN  NaN  NaN  NaN   
     4                 NaN          NaN  ...  NaN  NaN  NaN  NaN  NaN  NaN   
1971 1                 NaN          NaN  ...  NaN  NaN  NaN  NaN  NaN  NaN   

ID            606  607  608  609  
YEAR QUARTER                      
1968 4        NaN  NaN  NaN  NaN  
1969 1        NaN  NaN  NaN  NaN  
     2        NaN  NaN  NaN  NaN  
     3        NaN  NaN  NaN  NaN  
     4        NaN  NaN  NaN  NaN  
1970 1        NaN  NaN  NaN  NaN  
     2        NaN  NaN  NaN  NaN  
     3        NaN  NaN  NaN  NaN  
     4        NaN  NaN  NaN  NaN  
1971 1        NaN  NaN  NaN  NaN  

[10 rows x 349 columns]

In [61]:
# now we can calculate the avarage squared error for each forecaster and we can rank the forecasters based on this. 
mean_squared_errors = errors_df.mean()
ranked_forecasters = mean_squared_errors.sort_values()
print("Ranked forecasters based on mean squared error for NGDP3:")
print(ranked_forecasters)


Ranked forecasters based on mean squared error for NGDP3:
ID
150    1.098724e+03
24     8.291795e+03
71     8.751115e+03
6      9.098744e+03
29     9.260076e+03
           ...     
587    9.266672e+05
584    9.552285e+05
585    1.098955e+06
588    1.157768e+06
601    1.403110e+06
Length: 349, dtype: float64


In [62]:
import sys
sys.path.insert(0, '.')
from Bootstrap import rank_confidence_intervals_bootstrap

min_obs = 20
enough  = errors_df.columns[errors_df.notna().sum() >= min_obs]
print(f"Forecasters with >= {min_obs} obs: {len(enough)}")

X_wide = errors_df[enough]
print(f"Before dropna — shape: {X_wide.shape}")

X_wide = X_wide.dropna()
print(f"After dropna  — shape: {X_wide.shape}")


Forecasters with >= 20 obs: 149
Before dropna — shape: (228, 149)
After dropna  — shape: (0, 149)


If After dropna gives 0 rows, the forecasters never all overlap in the same period. The fix is to take only the top N most frequent forecasters.

In [63]:
# Take top N forecasters by number of observations
N = 30  # tune this until After dropna has enough rows
top_ids = errors_df.notna().sum().nlargest(N).index
X_wide  = errors_df[top_ids].dropna()
print(f"Top-{N} forecasters, {X_wide.shape[0]} complete periods")

X = X_wide.values



Top-30 forecasters, 0 complete periods


In [64]:
for N in [3, 5, 8, 10, 15, 20]:
    top_ids  = errors_df.notna().sum().nlargest(N).index
    complete = errors_df[top_ids].dropna().shape[0]
    print(f"N={N:3d}: {complete} complete periods")


N=  3: 53 complete periods
N=  5: 46 complete periods
N=  8: 22 complete periods
N= 10: 4 complete periods
N= 15: 1 complete periods
N= 20: 0 complete periods


# Simultaneous CI

## 1. Bootstrap Solution

In [82]:
import sys
sys.path.insert(0, '.')
from Bootstrap import rank_confidence_intervals_bootstrap


N       = 8
top_ids = errors_df.notna().sum().nlargest(N).index
X_wide  = errors_df[top_ids].dropna()
X       = X_wide.values  # shape (22, 8), already squared errors


population_ids = X_wide.columns.tolist()
n, p           = X.shape
print(f"Data shape: {n} time periods × {p} populations")

out = rank_confidence_intervals_bootstrap(X, alpha=0.05, B=20000, seed=42)



results_simbootstrap = pd.DataFrame({
    "Population" : population_ids,
    "Mean"       : out["theta_hat"].round(2),
    "CI_Lower"   : out["rank_ci"][:, 0],
    "CI_Upper"   : out["rank_ci"][:, 1],
    "CI_Width"   : out["rank_ci"][:, 1] - out["rank_ci"][:, 0],
})

results_simbootstrap = results_simbootstrap.sort_values("Mean", ascending=False).reset_index(drop=True)
results_simbootstrap.insert(0, "Est_Rank", results_simbootstrap.index + 1)

print("\nRank | Population | Mean       | 95% Rank CI  | CI Width")
print("-" * 58)
for _, row in results_simbootstrap.iterrows():
    print(
        f"  {int(row.Est_Rank):<4}"
        f"  {str(row.Population):<12}"
        f"  {row.Mean:<12,.1f}"
        f"  [{int(row.CI_Lower)}, {int(row.CI_Upper)}]"
        f"       {int(row.CI_Width)}"
    )


print(f"Bootstrap critical value: {out['critical_value']:.3f}\n")


Data shape: 22 time periods × 8 populations

Rank | Population | Mean       | 95% Rank CI  | CI Width
----------------------------------------------------------
  1     65.0          331,218.5     [1, 7]       6
  2     426.0         319,657.3     [1, 7]       6
  3     411.0         310,621.2     [1, 7]       6
  4     433.0         303,701.3     [1, 8]       7
  5     421.0         293,552.7     [1, 8]       7
  6     428.0         291,917.6     [1, 8]       7
  7     84.0          289,922.3     [1, 8]       7
  8     40.0          282,969.8     [4, 8]       4
Bootstrap critical value: 4.066



## 2. Simulation Solution

In [83]:
import sys
sys.path.insert(0, '.')
from Simulation import rank_confidence_intervals_simulation

N       = 8
top_ids = errors_df.notna().sum().nlargest(N).index
X_wide  = errors_df[top_ids].dropna()
X       = X_wide.values  # shape (22, 8), already squared errors


population_ids = X_wide.columns.tolist()
n, p           = X.shape
print(f"Data shape: {n} time periods × {p} populations")

out = rank_confidence_intervals_simulation(X, alpha=0.05, B=20000, seed=42)



results_simsimulation = pd.DataFrame({
    "Population" : population_ids,
    "Mean"       : out["theta_hat"].round(2),
    "CI_Lower"   : out["rank_ci"][:, 0],
    "CI_Upper"   : out["rank_ci"][:, 1],
    "CI_Width"   : out["rank_ci"][:, 1] - out["rank_ci"][:, 0],
})

results_simsimulation = results_simsimulation.sort_values("Mean", ascending=False).reset_index(drop=True)
results_simsimulation.insert(0, "Est_Rank", results_simsimulation.index + 1)

print("\nRank | Population | Mean       | 95% Rank CI  | CI Width")
print("-" * 58)
for _, row in results_simsimulation.iterrows():
    print(
        f"  {int(row.Est_Rank):<4}"
        f"  {str(row.Population):<12}"
        f"  {row.Mean:<12,.1f}"
        f"  [{int(row.CI_Lower)}, {int(row.CI_Upper)}]"
        f"       {int(row.CI_Width)}"
    )

print(f"Simulation critical value: {out['critical_value']:.3f}\n")

Data shape: 22 time periods × 8 populations

Rank | Population | Mean       | 95% Rank CI  | CI Width
----------------------------------------------------------
  1     65.0          331,218.5     [1, 3]       2
  2     426.0         319,657.3     [1, 5]       4
  3     411.0         310,621.2     [1, 6]       5
  4     433.0         303,701.3     [2, 8]       6
  5     421.0         293,552.7     [2, 8]       6
  6     428.0         291,917.6     [3, 8]       5
  7     84.0          289,922.3     [4, 8]       4
  8     40.0          282,969.8     [4, 8]       4
Simulation critical value: 2.991



### Comparison of two solutions

In [86]:
# compare the result of the two methods
comparison = results_simbootstrap.rename(columns={'CI_Lower': 'boot_lower', 'CI_Upper': 'boot_upper'}).merge(
    results_simsimulation.rename(columns={'CI_Lower': 'sim_lower', 'CI_Upper': 'sim_upper'})[['Population','sim_lower','sim_upper']],
    on='Population'
)


print("\nRank | Population | Mean       | CI Bootstrap | CI Simulation | ")
print("-" * 68)
for _, row in comparison.iterrows():
    print(
        f"  {int(row.Est_Rank):<4}"
        f"  {str(row.Population):<12}"
        f"  {row.Mean:<12,.1f}"
        f"  [{int(row.boot_lower)}, {int(row.boot_upper)}]"
        f"  [{int(row.sim_lower)}, {int(row.sim_upper)}]"
    )


Rank | Population | Mean       | CI Bootstrap | CI Simulation | 
--------------------------------------------------------------------
  1     65.0          331,218.5     [1, 7]  [1, 3]
  2     426.0         319,657.3     [1, 7]  [1, 5]
  3     411.0         310,621.2     [1, 7]  [1, 6]
  4     433.0         303,701.3     [1, 8]  [2, 8]
  5     421.0         293,552.7     [1, 8]  [2, 8]
  6     428.0         291,917.6     [1, 8]  [3, 8]
  7     84.0          289,922.3     [1, 8]  [4, 8]
  8     40.0          282,969.8     [4, 8]  [4, 8]


# The Stepwise Improvement

In [ ]:
from Stepwise_bootstrap import rank_ci_stepwise


In [87]:
# ── apply to your data ────────────────────────────────────────────────────────

X              = X_wide.values.astype(float)
population_ids = X_wide.columns.tolist()
n, p           = X.shape
print(f"Data shape: {n} time periods × {p} populations")

out = rank_ci_stepwise(X, alpha=0.05, B=5000, seed=42)

# ── results table ─────────────────────────────────────────────────────────────

results_stepbootstrap = pd.DataFrame({
    "Population" : population_ids,
    "Mean"       : out["theta_hat"].round(2),
    "CI_Lower"   : out["rank_ci"][:, 0],
    "CI_Upper"   : out["rank_ci"][:, 1],
    "CI_Width"   : out["rank_ci"][:, 1] - out["rank_ci"][:, 0],
})

results_stepbootstrap = results_stepbootstrap.sort_values("Mean", ascending=False).reset_index(drop=True)
results_stepbootstrap.insert(0, "Est_Rank", results_stepbootstrap.index + 1)

print("\nRank | Population | Mean       | 95% Rank CI  | CI Width")
print("-" * 58)
for _, row in results_stepbootstrap.iterrows():
    print(
        f"  {int(row.Est_Rank):<4}"
        f"  {str(row.Population):<12}"
        f"  {row.Mean:<12,.1f}"
        f"  [{int(row.CI_Lower)}, {int(row.CI_Upper)}]"
        f"       {int(row.CI_Width)}"
    )

Data shape: 22 time periods × 8 populations

Rank | Population | Mean       | 95% Rank CI  | CI Width
----------------------------------------------------------
  1     65.0          331,218.5     [1, 3]       2
  2     426.0         319,657.3     [1, 5]       4
  3     411.0         310,621.2     [1, 6]       5
  4     433.0         303,701.3     [2, 7]       5
  5     421.0         293,552.7     [2, 8]       6
  6     428.0         291,917.6     [3, 8]       5
  7     84.0          289,922.3     [4, 8]       4
  8     40.0          282,969.8     [5, 8]       3


In [88]:
# compare the results of simultaneous bootstrap and stepwise bootstrap
comparison = results_simbootstrap.rename(columns={'CI_Lower': 'boot_lower', 'CI_Upper': 'boot_upper'}).merge(
    results_stepbootstrap.rename(columns={'CI_Lower': 'step_lower', 'CI_Upper': 'step_upper'})[['Population','step_lower','step_upper']],
    on='Population'
)
print("\nRank | Population | Mean       | CI Bootstrap | CI Stepwise  | ")
print("-" * 68)
for _, row in comparison.iterrows():
    print(
        f"  {int(row.Est_Rank):<4}"
        f"  {str(row.Population):<12}"
        f"  {row.Mean:<12,.1f}"
        f"  [{int(row.boot_lower)}, {int(row.boot_upper)}]"
        f"  [{int(row.step_lower)}, {int(row.step_upper)}]"
    )


Rank | Population | Mean       | CI Bootstrap | CI Stepwise  | 
--------------------------------------------------------------------
  1     65.0          331,218.5     [1, 7]  [1, 3]
  2     426.0         319,657.3     [1, 7]  [1, 5]
  3     411.0         310,621.2     [1, 7]  [1, 6]
  4     433.0         303,701.3     [1, 8]  [2, 7]
  5     421.0         293,552.7     [1, 8]  [2, 8]
  6     428.0         291,917.6     [1, 8]  [3, 8]
  7     84.0          289,922.3     [1, 8]  [4, 8]
  8     40.0          282,969.8     [4, 8]  [5, 8]
